In [ ]:
import pandas as pd

from PySide6.QtCore import Qt
from PySide6.QtGui import QStandardItemModel, QStandardItem
from PySide6.QtWidgets import QDialog, QVBoxLayout, QLabel, QPushButton, QHBoxLayout
from PySide6.QtWidgets import QFileDialog

class HoldContinueDialog(QDialog):
    """
    - 보류: 다이얼로그를 닫지 않음(계속 멈춤)
    - 계속: accept()로 닫히고 다음 코드 진행
    """
    def __init__(self, parent=None, message=""):
        super().__init__(parent)
        self.setWindowTitle("확인 필요")
        self.setModal(True)
        self.setMinimumWidth(520)

        self.label = QLabel(message)
        self.label.setWordWrap(True)

        self.hold_btn = QPushButton("보류")
        self.continue_btn = QPushButton("계속")

        self.hold_btn.clicked.connect(self.on_hold)
        self.continue_btn.clicked.connect(self.accept)  # 계속 누르면 닫힘(진행)

        btn_layout = QHBoxLayout()
        btn_layout.addStretch(1)
        btn_layout.addWidget(self.hold_btn)
        btn_layout.addWidget(self.continue_btn)

        layout = QVBoxLayout(self)
        layout.addWidget(self.label)
        layout.addLayout(btn_layout)


    def on_hold(self):
        # 보류 누르면 "멈춘 상태" 유지: 다이얼로그를 닫지 않고 안내 문구만 바꿈
        self.label.setText(self.label.text() + "\n\n[보류중] 확인 후 '계속'을 눌러 진행하세요.")
        self.hold_btn.setEnabled(False)  # 중복 클릭 방지 (원하면 제거)


def fill_send_list(list_view, df_rows):
    """
    list_view: Qt Designer의 ListView (objectName: send_list) -> 보통 QListView
    df_rows: 조건에 맞는 행들 DataFrame (P,K,Q만 포함)
    """
    model = QStandardItemModel(list_view)

    for _, r in df_rows.iterrows():
        p = "" if pd.isna(r["P"]) else str(r["P"])
        k = "" if pd.isna(r["K"]) else str(r["K"])
        q = "" if pd.isna(r["Q"]) else str(r["Q"])

        # ListView는 기본적으로 1열 표시라서 한 줄 텍스트로 합쳐서 넣는 방식 추천
        text = f"P: {p} | K: {k} | Q: {q}"
        item = QStandardItem(text)
        item.setEditable(False)
        model.appendRow(item)

    list_view.setModel(model)

def on_click_open_excel(self):
    file_path, _ = QFileDialog.getOpenFileName(
        self,
        "엑셀 파일 선택",
        r"C:\Users\%USERNAME%\Downloads",   # 시작 폴더(원하면 변경)
        "Excel Files (*.xlsx *.xls)"
    )
    if not file_path:
        return  # 취소하면 아무것도 안함
    
    self.run_your_flow(file_path)
    
def check_onchannel_jeju_island_and_pause(parent_widget, send_list, df):
    """
    조건:
      - K열 == '온채널'
      - AQ열 > 1000
    있으면:
      - send_list에 P,K,Q 표시
      - 보류/계속 팝업 띄우고, 계속 눌러야 다음 코드 진행
    """
    # 엑셀 컬럼명을 이미 갖고 있다고 가정. (예: df.columns에 'K','P','Q','AQ'가 존재)
    # 만약 컬럼명이 없고 인덱스로만 다뤄야 하면 아래 주석 참고.

    mask = (df["K"].astype(str).str.strip() == "온채널") & (pd.to_numeric(df["AQ"], errors="coerce") > 1000)

    if mask.any():
        target = df.loc[mask, ["P", "K", "Q"]].copy()
        fill_send_list(send_list, target)

        dlg = HoldContinueDialog(
            parent=parent_widget,
            message="온채널 주문건에서 제주 또는 도서산간 주소지가 확인 되었습니다. 계속 진행 하시겠습니까?"
        )
        dlg.exec()  # 여기서 멈춤. '계속'을 누르면 다음 줄로 진행됨.

    # 조건에 해당 없거나, 계속 눌러서 닫혔으면 그냥 통과


# -------------------------------
# 사용 예시 (버튼 클릭 핸들러 내부 등)
# -------------------------------
def run_your_flow(self, excel_path):
    df = pd.read_excel(excel_path)

    # (중요) 여기서 조건 체크 & 필요 시 멈춤
    check_onchannel_jeju_island_and_pause(self, self.send_list, df)

    # 계속을 눌러야 여기부터 실행됨
    # ↓↓↓ 다음 작업 코드들
    # self.do_next_step()

In [4]:
# Jupyter에서 바로 실행 가능한 PySide6 테스트 코드 (한 셀)
import sys
import pandas as pd

from PySide6.QtCore import Qt
from PySide6.QtGui import QStandardItemModel, QStandardItem
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QPushButton, QLabel,
    QFileDialog, QDialog, QHBoxLayout, QListView
)


class HoldContinueDialog(QDialog):
    """보류: 닫지 않고 멈춤 / 계속: accept로 닫고 진행"""
    def __init__(self, parent=None, message=""):
        super().__init__(parent)
        self.setWindowTitle("확인 필요")
        self.setModal(True)
        self.setMinimumWidth(560)

        self.label = QLabel(message)
        self.label.setWordWrap(True)

        self.hold_btn = QPushButton("보류")
        self.continue_btn = QPushButton("계속")

        self.hold_btn.clicked.connect(self.on_hold)
        self.continue_btn.clicked.connect(self.accept)

        btn_layout = QHBoxLayout()
        btn_layout.addStretch(1)
        btn_layout.addWidget(self.hold_btn)
        btn_layout.addWidget(self.continue_btn)

        layout = QVBoxLayout(self)
        layout.addWidget(self.label)
        layout.addLayout(btn_layout)

    def on_hold(self):
        # 다이얼로그를 닫지 않으니 "멈춘 상태" 유지됨
        if "[보류중]" not in self.label.text():
            self.label.setText(self.label.text() + "\n\n[보류중] 확인 후 '계속'을 눌러 진행하세요.")
        self.hold_btn.setEnabled(False)


def fill_listview(list_view: QListView, df_rows: pd.DataFrame):
    model = QStandardItemModel(list_view)
    for _, r in df_rows.iterrows():
        p = "" if pd.isna(r["P"]) else str(r["P"])
        k = "" if pd.isna(r["K"]) else str(r["K"])
        q = "" if pd.isna(r["Q"]) else str(r["Q"])
        text = f"P: {p} | K: {k} | Q: {q}"
        item = QStandardItem(text)
        item.setEditable(False)
        model.appendRow(item)
    list_view.setModel(model)


def ensure_columns_by_letter(df: pd.DataFrame) -> pd.DataFrame:
    """
    엑셀 컬럼명이 'K','P','Q','AQ'로 되어있지 않은 경우 대비.
    엑셀 열 문자 기준 인덱스로 강제 매핑:
      K=11번째(0-based 10), P=16번째(15), Q=17번째(16), AQ=43번째(42)
    """
    needed = {"K": 10, "P": 15, "Q": 16, "AQ": 42}
    if len(df.columns) <= max(needed.values()):
        raise ValueError("엑셀 컬럼 수가 부족합니다. K/P/Q/AQ 위치를 확인하세요.")

    # 이미 컬럼명이 정확히 있으면 그대로 사용
    if all(col in df.columns for col in needed.keys()):
        return df

    # 없으면 위치로 rename해서 통일
    rename_map = {df.columns[idx]: name for name, idx in needed.items()}
    return df.rename(columns=rename_map)


def check_condition_and_pause(parent, send_list: QListView, df: pd.DataFrame) -> bool:
    """
    조건:
      - K == '온채널'
      - AQ > 1000
    있으면 팝업 실행(보류/계속) + 리스트 표시
    계속 누르면 True, 조건 없으면 True
    """
    df = ensure_columns_by_letter(df)

    k_series = df["K"].astype(str).str.strip()
    aq_series = pd.to_numeric(df["AQ"], errors="coerce")

    mask = (k_series == "온채널") & (aq_series > 1000)

    if mask.any():
        target = df.loc[mask, ["P", "K", "Q"]].copy()
        fill_listview(send_list, target)

        dlg = HoldContinueDialog(
            parent=parent,
            message="온채널 주문건에서 제주 또는 도서산간 주소지가 확인 되었습니다. 계속 진행 하시겠습니까?"
        )
        dlg.exec()  # '계속' 눌러야 다음 줄로 진행됨
    return True


class TestWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("엑셀 조건 체크 테스트")
        self.resize(720, 420)

        layout = QVBoxLayout(self)

        self.info = QLabel("1) [엑셀 선택] → 2) 조건(K=온채널 & AQ>1000) 있으면 팝업/리스트 표시 → 3) 계속 누르면 다음 단계 진행")
        self.info.setWordWrap(True)

        self.btn = QPushButton("엑셀 선택")
        self.btn.clicked.connect(self.open_excel)

        self.send_list = QListView()
        self.send_list.setMinimumHeight(220)

        self.log = QLabel("로그: -")
        self.log.setWordWrap(True)

        layout.addWidget(self.info)
        layout.addWidget(self.btn)
        layout.addWidget(QLabel("조건에 맞는 행(P/K/Q) 표시:"))
        layout.addWidget(self.send_list)
        layout.addWidget(self.log)

    def open_excel(self):
        path, _ = QFileDialog.getOpenFileName(self, "엑셀 파일 선택", "", "Excel Files (*.xlsx *.xls)")
        if not path:
            self.log.setText("로그: 파일 선택 취소")
            return

        self.log.setText(f"로그: 선택된 파일\n{path}")

        try:
            df = pd.read_excel(path)
        except Exception as e:
            self.log.setText(f"로그: 엑셀 읽기 실패\n{e}")
            return

        # 조건 체크 + (필요시) 보류/계속 팝업
        check_condition_and_pause(self, self.send_list, df)

        # 여기부터는 '계속' 눌렀거나 조건이 없었던 경우에만 도달
        self.log.setText(self.log.text() + "\n\n✅ 다음 단계 진행(여기에 후속 코드가 이어짐)")


# ---- 실행부 ----
app = QApplication.instance() or QApplication(sys.argv)
w = TestWindow()
w.show()

# 주피터에서 이벤트루프 진입
app.exec()

KeyboardInterrupt: 

0